# 02 — CU2: fine-tuning de BETO

Ajusta `dccuchile/bert-base-spanish-wwm-cased` (BETO) a las 21
categorías. Meta (OE2): F1-macro ≥ 0,85 en el conjunto de prueba y
superar al baseline del notebook 01.

Corre `entrenar_cu2_beto.py`: hasta 15 épocas con early stopping
(paciencia 3) sobre el F1-macro de validación, cabeza con LR 10x mayor
que el encoder, pérdida con pesos por clase + label smoothing, warmup +
decaimiento lineal y fp16 en GPU. Ver por qué en
`docs/MEJORA_MODELOS_IA.md`.

Tiempo estimado con GPU T4: del orden de 15-30 minutos.

In [ ]:
# Si ya corriste esta celda en esta sesión de Colab, puedes saltarla.
import os
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_ML = "/content/drive/MyDrive/NovaStore_IA/ml"
    get_ipython().system(f"pip install -q -r {CARPETA_ML}/requirements.txt")
else:
    # Ejecución local (verificación, no Colab): la carpeta ml/ del repo.
    CARPETA_ML = str(Path.cwd())

os.chdir(CARPETA_ML)
sys.path.insert(0, CARPETA_ML)
print("CARPETA_ML =", CARPETA_ML)


In [ ]:
MODO_RAPIDO = False  # True = pocas filas y 1 época: solo para probar que corre (métricas sin valor)

if MODO_RAPIDO:
    # Si todavía no corriste 00_preparar_dataset, genera un dataset sintético
    # diminuto para poder probar la plomería (no sirve para medir nada).
    from _modo_rapido_fallback import generar_si_falta
    generar_si_falta(Path(CARPETA_ML) / "datos")

import torch
print("GPU disponible:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("Sin GPU: en Colab, activa Entorno de ejecución -> Cambiar tipo de entorno -> T4 GPU.")


## Entrenar y evaluar en el conjunto de PRUEBA

In [ ]:
get_ipython().system("python entrenar_cu2_beto.py" + (" --modo-rapido" if MODO_RAPIDO else " --reanudar"))


## Resultados

In [ ]:
import json
from IPython.display import Image as MostrarImagen, display

carpeta = Path(CARPETA_ML) / "modelos" / "cu2-beto-v1"
metricas = json.loads((carpeta / "metricas.json").read_text(encoding="utf-8"))
print({k: v for k, v in metricas.items() if k in ("accuracy", "f1_macro", "f1_weighted", "n")})
print("Por fuente:", json.dumps(metricas.get("por_fuente", {}), indent=2, ensure_ascii=False))
for nombre in ("curvas_entrenamiento.png", "matriz_confusion.png"):
    if (carpeta / nombre).exists():
        display(MostrarImagen(filename=str(carpeta / nombre)))
